# MNIST from first principles: a modern multilayer perceptron

**Lecture 12 · Notebook 02 · Core route: 95 minutes · Full laboratory: 150 minutes**

Can a computer recognize a handwritten digit from 784 measured pixel intensities? We will answer that real classification question with an inspectable multilayer perceptron (MLP), not a framework shortcut.

$$
\text{image}\to\text{features}\to\text{logits}\to\text{probabilities}
\to\text{decision}\to\text{error analysis}.
$$

The notebook modernizes the earlier network: ReLU hidden units, fan-in-aware initialization, raw logits, stable softmax cross-entropy, shuffled mini-batches, and held-out evidence.

## Learning objectives and how to use this notebook

After this laboratory, you should be able to:

1. explain what information flattening preserves and discards;
2. scale integer pixels and one-hot encode mutually exclusive labels;
3. track every matrix shape in a dense network;
4. derive the softmax--cross-entropy gradient $P-Y$;
5. motivate Xavier and He random-normal initialization from fan-in;
6. trace vectorized forward propagation and backpropagation;
7. train with reproducible mini-batches; and
8. interpret a confusion matrix and inspect individual mistakes.

Run the core cells in order. Pause at each **checkpoint** before revealing the next calculation. The package code is short enough to inspect in `src/rice_dsm/ml/multiclass.py`; the notebook explains *why* each contract exists.

## Kernel preflight: verify the process before the experiment

A notebook kernel is a long-running Python process. VS Code can remember a kernel from another project even after you open this repository. The next cell uses only Python's standard library, reports the interpreter actually running, and stops with exact recovery commands before any course-package import can fail mysteriously.

In [ ]:
import os
import sys
from pathlib import Path

root_hint = os.environ.get("RICE_DSM_PROJECT_ROOT")
search_roots = ([Path(root_hint)] if root_hint else []) + [Path.cwd(), *Path.cwd().parents]
course_root = next((path.resolve() for path in search_roots if (path / "pyproject.toml").is_file()), None)
if course_root is None:
    raise RuntimeError("Open the course repository folder in VS Code, then restart this notebook.")
expected_prefix = (course_root / ".venv").resolve()
actual_prefix = Path(sys.prefix).resolve()
print("Python executable:", sys.executable)
print("Active environment:", actual_prefix)
print("Expected environment:", expected_prefix)
if actual_prefix != expected_prefix:
    raise RuntimeError(
        "Wrong notebook kernel. In the repository terminal run:\n"
        "  uv sync --locked\n"
        "  uv run python scripts/setup_course.py\n"
        "Then select 'Rice DSM' in VS Code and restart the kernel."
    )
assert Path(sys.executable).is_file()

## 1. Why this matters: from scanned mail to a controlled experiment

MNIST grew from NIST handwriting collections and became a standard benchmark after LeCun and collaborators standardized 60,000 training and 10,000 test images. Each $28\times28$ grayscale image represents a digit from 0 through 9. The benchmark is historically important, but today it is a **mechanism laboratory**, not evidence that a model is ready for unconstrained handwriting, accessibility, finance, or safety-critical use.

Our versioned archive contains a class-balanced, deterministic subset of authentic MNIST: 2,400 training images and 600 held-out test images. The bounded sample makes the entire experiment run offline in CI. It also makes a scientific limit explicit: our accuracy estimate has more sampling uncertainty than one computed on the complete benchmark.

**Question.** How accurately can a small dense network classify these held-out digits, and what structure appears in its mistakes?

### Experimental contract before code

The rows used to update parameters and the rows used to evaluate decisions must remain distinct. In this bounded lesson, `train_*` names the 2,400 fitting observations and `test_*` names the 600 held-out observations. We will not tune the architecture after looking at test results. In a larger investigation, introduce a validation partition for tuning and expose the test partition only once.

Keep four objects conceptually separate throughout:

| object | role | expected shape |
|---|---|---:|
| `train_images` | raw fitting images | `(2400, 28, 28)` |
| `train_labels` | integer class identities | `(2400,)` |
| `train_X` | flattened, scaled features | `(2400, 784)` |
| one-hot targets for a batch | loss representation | `(batch_size, 10)` |

The first code step imports reusable tools. The following step loads and validates the versioned artifact. Separating them makes failures easier to locate.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

# The package contains reusable, tested mathematics. This notebook supplies
# the experimental question, visual evidence, and interpretation.
from rice_dsm.ml.multiclass import (
    MultilayerPerceptronClassifier,
    one_hot,
    softmax,
    softmax_cross_entropy,
)
from rice_dsm.paths import mnist_teaching_sample_path

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 438  # Record the seed so initialization and shuffling are reproducible.

### Load the versioned data artifact

`mnist_teaching_sample_path()` resolves from the installed course package rather than assuming that the kernel's working directory is the repository root. This is the same path discipline expected in production modules and scripts.

NumPy's `.npz` archive stores named arrays. The context manager closes the archive after the arrays are materialized. We validate shapes, dtypes, and label support immediately: if these assertions fail, stop before modeling.

In [ ]:
dataset_path = mnist_teaching_sample_path()
with np.load(dataset_path) as archive:
    # Keep pixels as compact uint8 measurements until preprocessing is explicit.
    train_images = archive["x_train"]
    train_labels = archive["y_train"].astype(int)
    test_images = archive["x_test"]
    test_labels = archive["y_test"].astype(int)

print("data artifact:", dataset_path)
print("train images:", train_images.shape, train_images.dtype)
print("test images: ", test_images.shape, test_images.dtype)

assert train_images.shape == (2_400, 28, 28)
assert test_images.shape == (600, 28, 28)
assert train_labels.shape == (2_400,)
assert test_labels.shape == (600,)
assert train_images.dtype == test_images.dtype == np.uint8
assert set(np.unique(train_labels)) == set(range(10))

## 2. Inspect before transforming

An image tensor has axes `(observation, row, column)`. Here a pixel is an unsigned 8-bit integer from 0 (black background) through 255 (bright stroke). Begin with a structural profile: number of observations, possible labels, class counts, and intensity range.

This is not ceremonial exploratory analysis. It tests assumptions required by the architecture: ten classes require ten logits; 28-by-28 images require 784 dense inputs; unexpected intensity ranges would invalidate the `/ 255` scaling rule.

In [ ]:
# Count class frequency without changing row order or touching the held-out data.
class_counts = pd.Series(train_labels).value_counts().sort_index()
profile = pd.Series(
    {
        "training observations": len(train_images),
        "held-out observations": len(test_images),
        "image height": train_images.shape[1],
        "image width": train_images.shape[2],
        "minimum intensity": int(train_images.min()),
        "maximum intensity": int(train_images.max()),
    },
    name="value",
)
display(profile.to_frame())
display(class_counts.rename("training observations").to_frame().T)

assert class_counts.nunique() == 1  # The committed teaching subset is balanced.
assert int(train_images.min()) == 0 and int(train_images.max()) == 255

### Inspect representative measurements

Now view one observation from each class. A label does not explain all visible variation: stroke width, translation, rotation, and writing style vary within a class. Those sources of variation are part of the learning problem.

The selection below is deterministic—it chooses the first occurrence of each label—so the notebook does not quietly present a different gallery on every run.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for digit, axis in enumerate(axes.flat):
    # flatnonzero returns every matching row; [0] makes our display rule explicit.
    image_index = np.flatnonzero(train_labels == digit)[0]
    axis.imshow(train_images[image_index], cmap="gray_r", vmin=0, vmax=255)
    axis.set_title(f"label = {digit}")
    axis.axis("off")

fig.suptitle("One authentic MNIST training image from each class", weight="bold")
plt.tight_layout()

## 3. Flattening and scaling are model-interface choices

A dense layer expects one feature vector per observation. NumPy uses row-major order by default, so flattening places row 0 first, then row 1, and so on:

$$
X_i\in\mathbb R^{28\times28}
\longmapsto
x_i=\operatorname{vec}(X_i)\in\mathbb R^{784}.
$$

No pixel value is lost, but the model is no longer told that particular features are spatial neighbors. That missing **inductive bias** motivates convolution in the next notebook.

Converting to `float64` before division matters: the network needs real-valued arithmetic. Dividing by 255 maps the known sensor range to $[0,1]$. Unlike a fitted standard scaler, this rule estimates no statistic from the data; nevertheless, inference must apply the identical rule.

In [ ]:
n_train, height, width = train_images.shape
n_pixels = height * width

# Each row remains one observation. Only the within-observation representation changes.
train_X = train_images.reshape(n_train, n_pixels).astype(np.float64) / 255.0
test_X = test_images.reshape(len(test_images), n_pixels).astype(np.float64) / 255.0

print("raw training tensor: ", train_images.shape)
print("model feature matrix:", train_X.shape)
print("scaled feature range:", (train_X.min(), train_X.max()))

assert n_pixels == 784
assert train_X.shape == (2_400, 784)
assert test_X.shape == (600, 784)
assert np.all((0.0 <= train_X) & (train_X <= 1.0))

### Verify what flattening preserved

A useful preprocessing test reconstructs an image. If reshaping and rescaling recover the original pixel matrix, flattening preserved its values and order. The line plot on the right shows what the dense layer receives: the same measurements, now indexed from 0 through 783 rather than arranged on a grid.

In [ ]:
reconstructed = train_X[0].reshape(height, width) * 255.0
np.testing.assert_allclose(reconstructed, train_images[0])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].imshow(train_images[0], cmap="gray_r", vmin=0, vmax=255)
axes[0].set(title=f"matrix: {height} × {width}", xticks=[], yticks=[])
axes[1].plot(train_X[0], linewidth=1)
axes[1].set(
    title="the same 784 values after flattening",
    xlabel="feature index",
    ylabel="scaled intensity",
    ylim=(-0.05, 1.05),
)
plt.tight_layout()

## 4. Class identities, one-hot targets, and probabilities

The integer label `7` names a class; its numerical distance from `6` or `0` has no modeling meaning. For ten mutually exclusive classes, one-hot encoding maps $y_i=k$ to the standard basis vector $e_k\in\mathbb R^{10}$:

$$
7\longmapsto(0,0,0,0,0,0,0,1,0,0).
$$

One-hot encoding is a representation used by the loss. It does not create ten observations and does not turn the task into ten unrelated binary problems. The number and order of columns are part of the interface contract.

**Checkpoint.** Why must a mini-batch with no digit 9 still have ten target columns?

In [ ]:
sample_labels = np.array([7, 2, 0])
sample_targets = one_hot(sample_labels, n_classes=10)

display(
    pd.DataFrame(
        sample_targets.astype(int),
        index=pd.Index(sample_labels, name="integer label"),
        columns=[f"class {digit}" for digit in range(10)],
    )
)

assert sample_targets.shape == (3, 10)
np.testing.assert_allclose(sample_targets.sum(axis=1), 1.0)
np.testing.assert_array_equal(sample_targets.argmax(axis=1), sample_labels)

### Logits are scores, not probabilities

The output layer emits ten unconstrained **logits** $z_i$. A logit may be negative and the logits need not sum to one. Softmax converts the whole vector—not each coordinate independently—into a categorical probability distribution:

$$
p_{i,k}=\frac{e^{z_{i,k}}}{\sum_{r=0}^{9}e^{z_{i,r}}},
\qquad \sum_k p_{i,k}=1.
$$

The predicted class is the index of the largest probability. Because softmax is order-preserving, it is also the index of the largest logit.

In [ ]:
sample_logits = np.array([[1.2, -0.4, 2.1]])
sample_probabilities = softmax(sample_logits)

print("logits:       ", sample_logits)
print("probabilities:", np.round(sample_probabilities, 3))
print("predicted class:", sample_probabilities.argmax(axis=1)[0])

assert sample_logits.shape == sample_probabilities.shape == (1, 3)
np.testing.assert_allclose(sample_probabilities.sum(axis=1), 1.0)
assert sample_logits.argmax(axis=1)[0] == sample_probabilities.argmax(axis=1)[0]

## 5. Architecture and forward propagation

We use widths $784\to64\to10$. For a batch of $m$ observations stored in rows,

$$
X\;(m\times784)
\xrightarrow{W^1,b^1}
Z^1\;(m\times64)
\xrightarrow{\operatorname{ReLU}}
A^1\;(m\times64)
\xrightarrow{W^2,b^2}
Z^2\;(m\times10).
$$

Equivalently,

$$
Z^1=XW^1+\mathbf 1(b^1)^\top,
\quad A^1=\max(0,Z^1),
\quad Z^2=A^1W^2+\mathbf 1(b^2)^\top.
$$

Here $W^1\in\mathbb R^{784\times64}$ and $W^2\in\mathbb R^{64\times10}$. Bias vectors broadcast across observations, never across features accidentally. The final layer stays linear because stable softmax cross-entropy consumes logits directly.

In [ ]:
# This schematic deliberately draws fewer circles than the true layer widths.
# Its purpose is to communicate transformations, not parameter count.
fig, axis = plt.subplots(figsize=(11, 4.4))
layer_x = [0.0, 1.5, 3.0, 4.5]
labels = ["28 × 28 image", "784 scaled pixels", "64 ReLU features", "10 logits"]
counts = [9, 13, 9, 10]
colors = ["#4C78A8", "#72B7B2", "#F2CF5B", "#E45756"]

for layer, (x_position, label, count, color) in enumerate(
    zip(layer_x, labels, counts, colors)
):
    for y_position in np.linspace(-1.5, 1.5, count):
        axis.scatter(
            x_position, y_position, s=180, color=color, edgecolor="white", zorder=3
        )
    axis.text(x_position, 2.0, label, ha="center", weight="bold")
    if layer and layer < 3:
        axis.text(x_position, -2.05, "dense affine map\n+ ReLU", ha="center", fontsize=9)

for left, right in zip(layer_x[1:-1], layer_x[2:], strict=False):
    axis.annotate(
        "", xy=(right - 0.2, 0), xytext=(left + 0.2, 0),
        arrowprops={"arrowstyle": "->", "linewidth": 2}
    )
axis.annotate(
    "flatten + scale", xy=(1.25, 0), xytext=(0.25, 0), ha="center",
    arrowprops={"arrowstyle": "->", "linewidth": 2}
)
axis.set(
    title="The bounded MNIST multilayer perceptron",
    xlim=(-0.5, 5), ylim=(-2.35, 2.4)
)
axis.axis("off")
plt.tight_layout()

## 6. Random-normal weights must respect fan-in

For $z_j=\sum_{r=1}^{d}w_{rj}a_r$, approximate independence and centering give

$$
\operatorname{Var}(z_j)\approx d\,\operatorname{Var}(w_{rj})\operatorname{Var}(a_r).
$$

Unscaled $N(0,1)$ weights make variance grow with fan-in $d$. ReLU removes roughly half of a symmetric signal, motivating **He normal** weights $N(0,2/d)$ for hidden layers. A linear output commonly begins with **Xavier normal** weights $N(0,1/d)$. These assumptions are approximations, so we inspect the realized parameters and activations instead of treating the formulas as guarantees.

Biases begin at zero. Independent random weights already give hidden units different computations; random biases are unnecessary for symmetry breaking here.

In [ ]:
# Construction initializes parameters but does not fit them.
model = MultilayerPerceptronClassifier(
    (784, 64, 10),
    learning_rate=0.12,
    random_state=RANDOM_STATE,
)

parameter_shapes = pd.DataFrame(
    {
        "parameter": ["W1", "b1", "W2", "b2"],
        "shape": [
            model.weights_[0].shape,
            model.biases_[0].shape,
            model.weights_[1].shape,
            model.biases_[1].shape,
        ],
        "scalar count": [
            model.weights_[0].size,
            model.biases_[0].size,
            model.weights_[1].size,
            model.biases_[1].size,
        ],
    }
)
display(parameter_shapes)
print("total trainable scalars:", parameter_shapes["scalar count"].sum())

assert model.weights_[0].shape == (784, 64)
assert model.biases_[0].shape == (64,)
assert model.weights_[1].shape == (64, 10)
assert model.biases_[1].shape == (10,)

### Audit the initial signal before training

Forward propagation with `retain_cache=True` exposes the quantities backpropagation will need. On 256 observations, we compare observed weight standard deviations with their targets, inspect hidden preactivation spread, and measure the active ReLU fraction. Values need not equal the theoretical targets exactly; initialization is random and its variance argument is approximate.

In [ ]:
initial_logits, (initial_activations, initial_z) = model.forward(
    train_X[:256], retain_cache=True
)

initialization_evidence = pd.DataFrame(
    {
        "quantity": [
            "W1 observed std", "W1 He target",
            "W2 observed std", "W2 Xavier target",
            "Z1 observed std", "A1 nonzero fraction",
        ],
        "value": [
            np.std(model.weights_[0]), np.sqrt(2 / 784),
            np.std(model.weights_[1]), np.sqrt(1 / 64),
            np.std(initial_z[0]), np.mean(initial_activations[1] > 0),
        ],
    }
)
display(initialization_evidence)

assert initial_z[0].shape == initial_activations[1].shape == (256, 64)
assert initial_logits.shape == (256, 10)
assert np.isfinite(initial_logits).all()

## 7. Softmax cross-entropy supplies the output sensitivity

For one observation with one-hot target $y$, categorical cross-entropy is

$$
\ell(z,y)=-\sum_{k=0}^{9}y_k\log p_k
=\log\!\left(\sum_r e^{z_r}\right)-\sum_k y_kz_k.
$$

Differentiating coordinate $z_j$ gives

$$
\frac{\partial\ell}{\partial z_j}
=\frac{e^{z_j}}{\sum_r e^{z_r}}-y_j=p_j-y_j.
$$

Thus the output sensitivity is probability minus one-hot target. This compact result depends on the coupled softmax--cross-entropy contract; it is not a generic derivative for every output activation and loss.

### Numerical stability: shift before exponentiating

Naively evaluating $e^{1001}$ overflows floating-point arithmetic. Softmax is invariant when the same constant is added to every logit, so stable code subtracts the row maximum before exponentiating. The fused loss uses the corresponding log-sum-exp identity and never computes `log(0)` from a rounded probability.

In [ ]:
extreme_logits = np.array([[1_000.0, 1_001.0, 999.0]])
shifted_logits = extreme_logits - extreme_logits.max(axis=1, keepdims=True)

extreme_target = one_hot(np.array([1]), n_classes=3)
stable_loss, stable_gradient = softmax_cross_entropy(
    extreme_logits, extreme_target
)

print("shifted logits:", shifted_logits)
print("stable probabilities:", np.round(softmax(extreme_logits), 5))
print("stable loss:", stable_loss)
print("dL/dlogits:", np.round(stable_gradient, 5))

assert np.isfinite(stable_loss)
np.testing.assert_allclose(
    softmax(extreme_logits), softmax(shifted_logits), rtol=1e-12
)

### Check one derivative independently

An analytical derivation and its implementation can share the same mistake. A central finite difference supplies an independent local check:

$$
\frac{\partial\ell}{\partial z_j}
\approx\frac{\ell(z+\varepsilon e_j)-\ell(z-\varepsilon e_j)}{2\varepsilon}.
$$

This is a debugging test, not a training algorithm: it needs two loss evaluations per checked coordinate and is sensitive to the choice of $\varepsilon$.

In [ ]:
epsilon = 1e-6
positive = extreme_logits.copy()
negative = extreme_logits.copy()
positive[0, 0] += epsilon
negative[0, 0] -= epsilon

loss_positive, _ = softmax_cross_entropy(positive, extreme_target)
loss_negative, _ = softmax_cross_entropy(negative, extreme_target)
finite_difference = (loss_positive - loss_negative) / (2 * epsilon)

print("analytical derivative:", stable_gradient[0, 0])
print("finite difference:     ", finite_difference)
np.testing.assert_allclose(
    stable_gradient[0, 0], finite_difference, rtol=1e-5
)

## 8. Backpropagation through the dense classifier

For a mini-batch of $m$ observations, averaging the loss gives

$$
\Delta^2=\frac{P-Y}{m}\in\mathbb R^{m\times10}.
$$

Reverse the forward graph:

$$
\nabla_{W^2}L=(A^1)^\top\Delta^2,
\qquad \nabla_{b^2}L=\mathbf1^\top\Delta^2,
$$
$$
\Delta^1=(\Delta^2(W^2)^\top)\odot\mathbf1[Z^1>0],
$$
$$
\nabla_{W^1}L=X^\top\Delta^1,
\qquad \nabla_{b^1}L=\mathbf1^\top\Delta^1.
$$

The transpose reverses connectivity; the Hadamard product applies each ReLU's local derivative. The factor $1/m$ belongs exactly once. Omitting it makes the effective step size grow with batch size; applying it twice makes gradients too small.

Backpropagation computes derivatives. Gradient descent decides how to use them. This separation lets the same backward pass support SGD, momentum, Adam, and other optimizers.

In [ ]:
# Use a small batch so every matrix shape is easy to inspect.
batch_X = train_X[:32]
batch_y = train_labels[:32]
batch_loss, weight_gradients, bias_gradients = model.loss_and_gradients(
    batch_X, batch_y
)

for layer_index, (weights, weight_gradient, bias_gradient) in enumerate(
    zip(model.weights_, weight_gradients, bias_gradients), start=1
):
    print(
        f"layer {layer_index}: "
        f"W {weights.shape}, dW {weight_gradient.shape}, "
        f"db {bias_gradient.shape}, ||dW|| = {np.linalg.norm(weight_gradient):.3f}"
    )

print("initial mini-batch loss:", round(batch_loss, 3))
assert all(w.shape == g.shape for w, g in zip(model.weights_, weight_gradients))
assert all(b.shape == g.shape for b, g in zip(model.biases_, bias_gradients))
assert all(np.isfinite(g).all() for g in weight_gradients + bias_gradients)

## 9. Mini-batches are a statistical and systems compromise

A full-batch update uses all 2,400 training observations; a stochastic update uses one. A mini-batch estimates the full gradient while supplying enough matrix work for efficient numerical kernels. Smaller batches introduce more update noise; larger batches require more memory and provide fewer updates per epoch.

The course implementation shuffles row indices each epoch, uses every observation once, and retains a smaller final batch when necessary. With 2,400 observations and batch size 64, one epoch performs $\lceil2400/64\rceil=38$ updates.

The test data below are recorded for a fixed teaching demonstration, not used for gradient updates. In model development, use validation data for choices such as width, learning rate, and stopping, then expose the test data once after those choices are frozen.

In [ ]:
# fit mutates model parameters and stores one set of aggregate metrics per epoch.
model.fit(
    train_X,
    train_labels,
    epochs=8,
    batch_size=64,
    validation_data=(test_X, test_labels),
)

history = pd.DataFrame(model.history_)
history.index = np.arange(1, len(history) + 1)
history.index.name = "epoch"
display(history.round(3))

assert len(history) == 8
assert history["loss"].iloc[-1] < history["loss"].iloc[0]
assert history["validation_accuracy"].iloc[-1] > 0.80

### Read the curves as two different kinds of evidence

Training cross-entropy is **optimization evidence**: did the algorithm reduce its fitting objective? Accuracy on observations excluded from updates is **decision evidence**: how often did the current decision rule succeed on those rows? A decreasing loss does not guarantee improving held-out accuracy, calibrated probabilities, or performance after distribution shift.

Because this lesson displays held-out accuracy every epoch, we must not pretend the final epoch was selected without looking. The graph teaches monitoring; a professional selection protocol would monitor a validation set and reserve the test set.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Loss and accuracy use different units, so separate panels are clearer than
# a dual-axis graph that could imply a direct numerical comparison.
axes[0].plot(history.index, history["loss"], marker="o", color="tab:purple")
axes[0].set(
    title="Optimization evidence",
    xlabel="epoch",
    ylabel="training cross-entropy",
)
axes[1].plot(history.index, history["accuracy"], marker="o", label="training")
axes[1].plot(
    history.index, history["validation_accuracy"], marker="o", label="held out"
)
axes[1].set(
    title="Decision evidence",
    xlabel="epoch",
    ylabel="accuracy",
    ylim=(0.65, 1.0),
)
axes[1].legend()
fig.suptitle("A loss curve is not a generalization claim", weight="bold")
plt.tight_layout()

## 10. Worked example: held-out decisions and asymmetric questions

Accuracy asks what fraction of decisions were correct. Precision asks, among observations predicted as class $k$, what fraction truly were $k$. Recall asks, among true class-$k$ observations, what fraction the model recovered. The F1 score harmonically combines precision and recall.

A confusion matrix retains the pair `(true class, predicted class)` for every decision. Even with balanced classes, it can reveal concentrated confusions hidden by overall accuracy. In an operational system, those confusions should be connected to domain-specific costs.

In [ ]:
# Freeze the trained model conceptually before this final evaluation step.
test_predictions = model.predict(test_X)
test_probabilities = model.predict_proba(test_X)

report = pd.DataFrame(
    classification_report(
        test_labels,
        test_predictions,
        output_dict=True,
        zero_division=0,
    )
).T
class_report = report.loc[
    [str(digit) for digit in range(10)],
    ["precision", "recall", "f1-score", "support"],
]
display(class_report.round(3))
print(f"held-out accuracy: {np.mean(test_predictions == test_labels):.3f}")

assert test_predictions.shape == (600,)
assert test_probabilities.shape == (600, 10)
np.testing.assert_allclose(test_probabilities.sum(axis=1), 1.0)

### Confusion counts preserve the error pairs

Rows are true digits and columns are predicted digits. Keep integer counts when estimating how much evidence or workload each cell represents. Row-normalized values answer a different question—conditional recall—and can make a rare class look as visually prominent as a common one.

In [ ]:
matrix = confusion_matrix(
    test_labels,
    test_predictions,
    labels=np.arange(10),  # Fix row/column order even if a class were absent.
)

fig, axis = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(matrix, display_labels=np.arange(10)).plot(
    ax=axis,
    cmap="Blues",
    colorbar=False,
)
axis.set_title("Held-out MNIST confusion counts")
plt.tight_layout()

assert matrix.shape == (10, 10)
assert matrix.sum() == len(test_labels)

## 11. Error analysis: move from aggregates back to observations

A confusion count does not reveal why an error occurred. Inspect mistakes together with the model's largest reported probability. High-confidence errors are especially diagnostic: they may reveal unusual handwriting, ambiguous labels, preprocessing mismatches, or systematic overconfidence.

Confidence is not calibration. Calibration is a population-level statement comparing predicted probabilities with empirical frequencies and requires a separate reliability analysis. Here confidence is only a ranking device for selecting cases to inspect.

In [ ]:
mistake_indices = np.flatnonzero(test_predictions != test_labels)
mistake_confidence = test_probabilities[mistake_indices].max(axis=1)

# Sort descending and show the ten mistaken observations the model trusted most.
ranking = np.argsort(mistake_confidence)[::-1]
shown = mistake_indices[ranking[:10]]

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for index, axis in zip(shown, axes.flat):
    axis.imshow(test_images[index], cmap="gray_r", vmin=0, vmax=255)
    axis.set_title(
        f"true {test_labels[index]} → pred {test_predictions[index]}\n"
        f"confidence {test_probabilities[index].max():.2f}"
    )
    axis.axis("off")

fig.suptitle("The model's most confident held-out mistakes", weight="bold")
plt.tight_layout()
assert len(mistake_indices) > 0

## 12. Debugging, common failure modes, tests, and professional practice

- **Loss becomes `nan`:** inspect pixel scale, learning rate, log-sum-exp stability, parameter norms, and nonfinite gradients.
- **Accuracy stays near 0.10:** verify row/label alignment, class order, output width, and whether an update changes parameters.
- **Loss falls but held-out accuracy stalls:** distinguish optimization from generalization; investigate capacity, leakage, and shift.
- **Every hidden unit is inactive:** inspect preactivations and reconsider input scale, initialization, and step size.
- **Results change between runs:** record seed, data identity, shuffle policy, environment, and dtype.
- **A broadcast succeeds with the wrong meaning:** assert every feature, activation, target, and gradient shape.

The notebook is an experimental narrative. Reusable mathematics lives in the tested package. A professional run records data provenance, split identity, code revision, preprocessing, class order, architecture, initializer, optimizer settings, seed, metrics, and artifacts. A saved model without its input and class-order contracts is incomplete.

In [ ]:
# These post-training invariants are inexpensive smoke tests. Package tests also
# check gradients and failure behavior on small deterministic examples.
diagnostics = {
    "all parameters finite": all(np.isfinite(w).all() for w in model.weights_),
    "all probabilities finite": np.isfinite(test_probabilities).all(),
    "probabilities normalized": np.allclose(test_probabilities.sum(axis=1), 1.0),
    "all classes predicted": len(np.unique(test_predictions)) == 10,
    "class order preserved": test_probabilities.shape[1] == 10,
}
display(pd.Series(diagnostics, name="passed"))
assert all(diagnostics.values())

## 13. Guided investigation: does width help this bounded experiment?

Compare hidden widths 16 and 64 while holding seed, learning rate, epochs, batch size, data, and evaluation fixed. This is a controlled comparison, not a definitive hyperparameter search. Width changes both representation capacity and parameter count.

Before running the cell, predict which model will attain lower training loss. Then ask a separate question: does lower training loss necessarily imply better held-out accuracy?

In [ ]:
width_evidence = []
for hidden_width in (16, 64):
    candidate = MultilayerPerceptronClassifier(
        (784, hidden_width, 10),
        learning_rate=0.12,
        random_state=RANDOM_STATE,
    ).fit(
        train_X,
        train_labels,
        epochs=4,
        batch_size=64,
        validation_data=(test_X, test_labels),
    )

    parameter_count = sum(
        weights.size + biases.size
        for weights, biases in zip(candidate.weights_, candidate.biases_)
    )
    width_evidence.append(
        {
            "hidden width": hidden_width,
            "parameters": parameter_count,
            "training loss": candidate.history_["loss"][-1],
            "held-out accuracy": candidate.history_["validation_accuracy"][-1],
        }
    )

width_table = pd.DataFrame(width_evidence)
display(width_table.round(3))
assert len(width_table) == 2
assert width_table["parameters"].is_monotonic_increasing

### Interpret the comparison before changing more variables

If the wider model performs better here, that observation does not prove that width always helps. It is conditional on this sample, seed, optimizer, and short training budget. If you next change epochs and learning rate simultaneously, you will no longer know which change caused a difference.

A professional experiment table records the hypothesis, controlled factors, changed factor, metric, and conclusion. Repeated model choice belongs on validation data; the comparison above reuses held-out data only as a compact classroom demonstration.

## 14. Independent extension exercise: Fashion-MNIST

Fashion-MNIST preserves the same 28-by-28, ten-class interface while replacing digits with clothing categories. Build an independent applied study:

1. state a decision question and obtain the dataset from its authoritative source;
2. preserve a test set before choosing architecture or learning rate;
3. verify shapes, range, class meanings, and licenses;
4. reuse the same flattening, scaling, and one-hot contracts;
5. compare at least two justified architectures or regularization choices using validation evidence;
6. report accuracy, per-class recall, a confusion matrix, and representative high-confidence errors;
7. discuss which clothing classes are visually ambiguous and what a dense model ignores spatially; and
8. add tests for every reusable package change.

Do not claim that an improved score proves commercial readiness. Document the population represented by the data and important populations it does not represent.

## Retrieval practice

1. What does flattening preserve, and what inductive bias does it remove?
2. Why is dividing by 255 different from fitting a standard scaler?
3. Why does one-hot width remain ten for every mini-batch?
4. Derive $\partial\ell/\partial z_j=p_j-y_j$ without memorizing it.
5. Why does the first hidden layer use $\sqrt{2/784}$ rather than standard-normal weights?
6. What evidence does a loss curve provide? What evidence does it not provide?
7. Why inspect confident mistakes after viewing a confusion matrix?

## Takeaway and further reading

An MLP turns flattened, scaled pixels into learned features, then logits, probabilities, and decisions. The modern output contract—linear logits plus stable softmax cross-entropy—gives the clean gradient $(P-Y)/m$. Fan-in-aware random-normal initialization keeps early signals usable; backpropagation supplies derivatives; shuffled mini-batches supply practical updates. Held-out aggregates begin evaluation, while classwise and observation-level error analysis make it informative.

Authoritative sources:

- LeCun, Cortes, and Burges, [The MNIST database](http://yann.lecun.com/exdb/mnist/).
- He et al. (2015), [Delving Deep into Rectifiers](https://doi.org/10.1109/ICCV.2015.123).
- Glorot and Bengio (2010), [Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a.html).
- NumPy, [random sampling and reproducibility](https://numpy.org/doc/stable/reference/random/index.html); Python's standard-library [random documentation](https://docs.python.org/3/library/random.html) clarifies that its generator is a different API.
- Xiao, Rasul, and Vollgraf (2017), [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist).